# Apply Gold Metadata

Applies repository-managed metadata from `Configs/metadata/` to the Gold consumption views in Unity Catalog.

This notebook:

- validates that configured metadata matches the deployed Gold view schemas
- applies view and column comments
- verifies that the persisted Unity Catalog metadata matches the repository configuration

The JSON metadata files are the source of truth for both Unity Catalog documentation and Genie table/column descriptions.

In [0]:
from pathlib import Path
import sys


# Locate repository root
current_path = Path.cwd().resolve()
repo_root = None

for candidate in [current_path] + list(current_path.parents):
    if (
        (candidate / "Configs" / "metadata").exists()
        and (candidate / "src" / "deployment").exists()
    ):
        repo_root = candidate
        break


if repo_root is None:
    raise RuntimeError(
        "Could not locate the repository root."
    )


# Make deployment modules importable
deployment_src = (
    repo_root
    / "src"
    / "deployment"
)

if str(deployment_src) not in sys.path:
    sys.path.insert(
        0,
        str(deployment_src)
    )


print(f"Repository root: {repo_root}")
print(f"Deployment source: {deployment_src}")

In [0]:
import importlib
import apply_gold_metadata

importlib.reload(apply_gold_metadata)

from apply_gold_metadata import (
    apply_gold_metadata,
    load_metadata_configs
)


# Dry-run validation before making any changes
apply_gold_metadata(
    spark=spark,
    dry_run=True
)

In [0]:
# Apply repository metadata to Unity Catalog
apply_gold_metadata(
    spark=spark,
    dry_run=False
)

In [0]:
# Validate persisted Unity Catalog metadata

metadata_configs = load_metadata_configs()

validation_errors = []


for metadata in metadata_configs:
    table_name = metadata["table"]
    catalog, schema, table = table_name.split(".")

    # Validate table comment
    table_result = spark.sql(
        f"""
        SELECT comment
        FROM `{catalog}`.information_schema.tables
        WHERE table_schema = '{schema}'
          AND table_name = '{table}'
        """
    ).collect()

    if not table_result:
        validation_errors.append(
            f"{table_name}: table not found in information_schema"
        )
        continue

    actual_table_comment = table_result[0]["comment"]
    expected_table_comment = metadata["description"]

    if actual_table_comment != expected_table_comment:
        validation_errors.append(
            f"{table_name}: table comment does not match metadata"
        )

    # Validate column comments
    column_rows = spark.sql(
        f"""
        SELECT column_name, comment
        FROM `{catalog}`.information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name = '{table}'
        """
    ).collect()

    actual_column_comments = {
        row["column_name"]: row["comment"]
        for row in column_rows
    }

    for column_name, column_info in metadata["columns"].items():
        expected_comment = column_info["description"]
        actual_comment = actual_column_comments.get(column_name)

        if actual_comment != expected_comment:
            validation_errors.append(
                f"{table_name}.{column_name}: "
                "comment does not match metadata"
            )

    print(
        f"{table_name}: "
        f"table comment + "
        f"{len(metadata['columns'])} column comments checked"
    )


if validation_errors:
    print("\nMetadata validation FAILED:")

    for error in validation_errors:
        print(f"- {error}")

    raise AssertionError(
        f"{len(validation_errors)} metadata validation error(s) found."
    )


print()
print("Persisted Unity Catalog metadata validation PASSED.")